In [1]:
import os
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

from case_study.utils import set_matplot_style, BI_PART_COLORS, process_metadata

from src.utils import load_env, get_logger, load_json, save_to_json
from src.experiment_config import ExperimentConfig
from src.data_loading import DatasetLoader

set_matplot_style()
env_vars = load_env()
logger = get_logger("case study")

/Users/aleto/projects/metaphor-detector/env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
config = ExperimentConfig(
    task_type="analysis",
    task_name="podcast_case_study",
    dataset="podcasts",
    logger=logger,
    env_vars=env_vars,
    skip_load=False
)
data_loader = DatasetLoader(config)

In [3]:
# LOAD DATA

# metaphor classification results so far
met_res_df = data_loader.load_metaphor_classification_results()
unique_docs = list(set(met_res_df["doc_id"].to_list()))


mets_df = data_loader.load_metaphor_classification_results(metaphors_only=True)
mets_df["doc_id"] = mets_df["doc_id"].apply(lambda x: "-".join((x.split("_")[0]).split("-")[:-1]))

# PRINT SUMMARY
print(f"Metaphors in podcast dataset: {len(mets_df)} from {len(mets_df["doc_id"].unique())} documents")  # 6313

2026-07-29 17:26:26,453 - case study - INFO - Loading qwen 'binary metaphor' (specified best model) classification results...
2026-07-29 17:26:26,785 - case study - INFO - Loading qwen 'binary metaphor' (specified best model) classification results...


Metaphors in podcast dataset: 18079 from 94 documents


In [4]:
# join the two so we can access metadata
pod_df = data_loader.load_raw_data()
pod_df["doc_id"] = pod_df["id"]
confirmed_mets_df = mets_df.merge(pod_df, on="doc_id", how="left")

In [5]:
confirmed_mets_grouped = confirmed_mets_df.groupby(by="doc_id").count().sort_values(by="sentence")
confirmed_mets_grouped["count"] = confirmed_mets_grouped["sentence"]
confirmed_mets_grouped = confirmed_mets_grouped[["count"]]
print(f"Processed {len(confirmed_mets_grouped)} documents")
confirmed_mets_grouped[confirmed_mets_grouped["count"] < 100]

Processed 94 documents


,count
doc_id,
876e1918-4bbd-11ec-9286-330a99840065,13
e8ec7bcb-06ae-47c3-adfd-09176416ff2a,65
6719607c-9932-11ee-9017-eb3e8a6583cb,72
553aff2e-8dfc-11ed-826a-131940f84f69,75
13b33b9c-d276-11ee-9b50-07cb70f0898b,87
b75ea1be-17e0-11ef-a17e-bb3416e4183b,89
db3645a8-866b-11ee-89ee-ab796eb80dd3,93


In [6]:
finised_df = confirmed_mets_grouped[confirmed_mets_grouped["count"] >= 100]
print(f"Completed processing {len(finised_df)} documents")

Completed processing 87 documents


In [7]:
not_enough_df = confirmed_mets_grouped[confirmed_mets_grouped["count"] < 100]
not_enough_df = not_enough_df[["count"]]
print(f"Not enough metaphors yet for {len(not_enough_df)} documents")
not_enough_df.head()

Not enough metaphors yet for 7 documents


,count
doc_id,
876e1918-4bbd-11ec-9286-330a99840065,13
e8ec7bcb-06ae-47c3-adfd-09176416ff2a,65
6719607c-9932-11ee-9017-eb3e8a6583cb,72
553aff2e-8dfc-11ed-826a-131940f84f69,75
13b33b9c-d276-11ee-9b50-07cb70f0898b,87


In [8]:
# get ones we haven't even processed
processed_doc_ids = confirmed_mets_grouped.index.to_list()

all_pod_ids = pod_df["doc_id"].tolist()
missing_ids = [pod_id for pod_id in all_pod_ids if pod_id not in processed_doc_ids]

missing_df = pd.DataFrame()
missing_df["doc_id"] = missing_ids
missing_df["sentence"] = 0
missing_df = missing_df.set_index("doc_id")
print(f"Missing {len(missing_df)} docs from processed set...")

Missing 0 docs from processed set...


In [9]:
to_process_df = pd.concat([missing_df, not_enough_df])
print(f"Need more metaphors for {len(to_process_df)} documents")

Need more metaphors for 7 documents


In [10]:
# read in candidates
candidate_path = f"{env_vars["RESULTS_DIR"]}/get_candidate_metaphors/rule_based/podcasts_metaphor_paths.json"
candidate_full = load_json(candidate_path)
candidate_data = candidate_full["data"]
print(f"Loaded {len(candidate_data)} candidates")

candidate_df = pd.DataFrame()
candidate_df["met_id"] = list(candidate_data.keys())

Loaded 140754 candidates


In [11]:
# read in '...with_ads.json' remove processed candidates
met_class_w_ads = f"{env_vars["RESULTS_DIR"]}/metaphor_classification/llm/podcasts_source_verb_target_noun_binary_met_class_qwen_with_ads.json"
processed_mets = load_json(met_class_w_ads)["data"]
processed_sdp_ids = list(processed_mets.keys())
print(f"Already processed {len(processed_sdp_ids)} candidates")

Already processed 56254 candidates


In [12]:
candidate_df_filtered = candidate_data
candidate_df["processed"] = candidate_df["met_id"].apply(lambda x: x in processed_sdp_ids)

In [13]:
candidate_df = candidate_df[candidate_df["processed"] == False]
print(f"Remaining candidates to process: {len(candidate_df)}")

Remaining candidates to process: 84500


In [14]:
# create a df of candidates with doc_id
candidate_df["chunk_id"] = candidate_df["met_id"].apply(lambda x: x.split("_")[0])
candidate_df["doc_id"] = candidate_df["chunk_id"].apply(lambda x: ("-").join(x.split('-')[:-1]))
candidate_df["to_process"] = candidate_df["doc_id"].apply(lambda x: x in to_process_df.index.to_list())
candidate_df = candidate_df[candidate_df["to_process"] == True]
print(f"Candidates from docs that need mets: {len(candidate_df)}")

Candidates from docs that need mets: 0


In [15]:
cadidate_df = candidate_df.drop(columns=["to_process"])
print(len(candidate_df["doc_id"].unique())) # why 58 instead of 59?
candidate_df.groupby("doc_id").count().sort_values(by="met_id")

0


,met_id,processed,chunk_id,to_process
doc_id,,,,


In [16]:
# sample for each that needs more (to_process_df)
candidate_df.head()

,met_id,processed,chunk_id,doc_id,to_process


In [17]:
N = 200  # number of rows to sample per doc_id
sampled_df = (
    candidate_df.groupby('doc_id', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), N), replace=False, random_state=42))
    .reset_index(drop=True)
)
sampled_df["doc_id"] = sampled_df["chunk_id"].apply(lambda x: ("-").join(x.split("-")[:-1]))
print(len(sampled_df["doc_id"].unique()))
sampled_met_ids = sampled_df["met_id"].to_list()
print(len(sampled_met_ids))

sampled_df.groupby("doc_id").count().sort_values("met_id")

0
0


,met_id,processed,chunk_id,to_process
doc_id,,,,


In [18]:
# construct the dict again
sampled_data = {k: v for k, v in candidate_data.items() if k in sampled_met_ids}

In [19]:
print(f"saving {len(sampled_data)} candidates")
save_path = f"{env_vars['RESULTS_DIR']}/get_candidate_metaphors/rule_based"
filename = "podcasts_metaphor_paths-sampled.json"
candidate_full["data"] = sampled_data
save_to_json(candidate_full, save_path, filename)

saving 0 candidates


In [20]:
print(len(sampled_data))

0
